<a href="https://colab.research.google.com/github/nejumi/wandb-physicalai-workshop/blob/main/notebooks/02_isaac_lab_wandb_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Isaac Lab × W&B — Cartpole PPO quickstart

Isaac Lab の standalone Newton backend で `Isaac-Cartpole` を短く学習します。GUI 版 Isaac Sim は使いません。

完了時には同じ W&B Run に次の3種類が揃います。

- Metrics: reward、episode length、PPO loss
- Model Artifact: `isaaclab-cartpole-policy`
- Video: `eval/simulation`

Colab の **ランタイム > ランタイムのタイプを変更 > T4 GPU** を選び、上から順に実行してください。初回セットアップは5〜10分、学習は2〜3分が目安です。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
import sys
print('Python:', sys.version)

## 1. 教材と Isaac Lab を準備する

Isaac Lab は commit に固定し、専用の Python 3.12 環境へインストールします。Colab カーネルが Python 3.13 でも学習はこの専用環境で実行します。大量の依存を取得するため、このセルが最も時間を使います。

検証範囲: ローカル NVIDIA GPU で確認しています。Google Colab T4 実環境での全セル完走は未確認です。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Install into the running kernel's Python; do not assume a home directory or PATH.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.13.8'], check=True)
uv = Path(subprocess.check_output(
    [sys.executable, '-c', 'import uv; print(uv.find_uv_bin())'], text=True
).strip())
subprocess.run([str(uv), '--version'], check=True)
os.environ['UV_BIN'] = str(uv)
os.environ['PATH'] = str(uv.parent) + os.pathsep + os.environ['PATH']

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

workdir = Path(os.environ.get('PHYSICAL_AI_WORKDIR', '/content'))
workdir.mkdir(parents=True, exist_ok=True)
workshop = workdir / 'wandb-physicalai-workshop'
isaaclab = workdir / 'IsaacLab'
if not (workshop / '.git').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nejumi/wandb-physicalai-workshop.git', str(workshop)], check=True)
else:
    subprocess.run(['git', '-C', str(workshop), 'pull', '--ff-only'], check=True)
if not (isaaclab / '.git').exists():
    subprocess.run(['git', 'clone', '-q', '--no-checkout', 'https://github.com/isaac-sim/IsaacLab.git', str(isaaclab)], check=True)
isaaclab_commit = '7bca8097a5ed27c26744686c84355d1786769e81'
subprocess.run(['git', '-C', str(isaaclab), 'fetch', '-q', '--depth', '1', 'origin', isaaclab_commit], check=True)
subprocess.run(['git', '-C', str(isaaclab), 'checkout', '-q', isaaclab_commit], check=True)
subprocess.run([str(uv), 'sync', '--python', '3.12', '--locked', '--extra', 'wandb', '--extra', 'video'], cwd=isaaclab, check=True)

## 2. W&B の接続情報を設定する

Colab Secrets の `WANDB_API_KEY` を優先し、なければ入力します。Dedicated Cloud の場合は `WANDB_BASE_URL` も登録してください。

In [ ]:
import getpass, os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

os.environ['WANDB_API_KEY'] = secret('WANDB_API_KEY') or getpass.getpass('WANDB_API_KEY: ')
os.environ['WANDB_BASE_URL'] = (secret('WANDB_BASE_URL') or 'https://api.wandb.ai').rstrip('/')
os.environ['WANDB_PROJECT'] = secret('WANDB_PROJECT') or 'physical-ai-workshop'
entity = secret('WANDB_ENTITY')
if entity:
    os.environ['WANDB_ENTITY'] = entity
print('project:', os.environ['WANDB_PROJECT'])
print('WANDB_MODE:', os.environ.get('WANDB_MODE', 'online (default)'))

## 3. Cartpole PPO を2 iterationsだけ実行する

初回は CUDA kernel compile に時間がかかります。Isaac Lab native logger が学習曲線を記録し、学習後に同じ Run を resume して最新 checkpoint と mp4 を追加します。

In [ ]:
import subprocess, uuid

os.environ['WANDB_RUN_ID'] = uuid.uuid4().hex[:12]
os.environ['ISAACLAB_DIR'] = str(isaaclab)
os.environ['ISAACLAB_ITERATIONS'] = '2'
os.environ['ISAACLAB_NUM_ENVS'] = '32'
script = workshop / 'scripts' / '10_run_isaac_colab.sh'
if not script.is_file():
    raise FileNotFoundError(f'Workshop script not found: {script}')
with subprocess.Popen(['bash', str(script)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED='1')) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)

## W&B Run を開く

このセルは学習を実行しません。学習済み Run の URL を取得して表示します。旧版で学習済みの場合も、このセルだけ実行できます。

In [ ]:
import html, json, os, subprocess, sys
from pathlib import Path
from IPython.display import HTML, display

workdir = Path(os.environ.get('PHYSICAL_AI_WORKDIR', '/content'))
workshop = workdir / 'wandb-physicalai-workshop'
isaaclab = workdir / 'IsaacLab'
helper = workshop / 'scripts' / '11_show_isaac_run.py'
if not helper.is_file():
    subprocess.run(['git', '-C', str(workshop), 'pull', '--ff-only'], check=True, capture_output=True, text=True)
# Use the installed Isaac environment directly; no uv or training needed.
command = [str(isaaclab / '.venv/bin/python'), str(helper), '--isaaclab', str(isaaclab)]
if os.environ.get('WANDB_RUN_ID'):
    command += ['--run-id', os.environ['WANDB_RUN_ID']]
result = subprocess.run(command, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr)
run_info = json.loads(result.stdout)
if run_info.get('url'):
    url = html.escape(run_info['url'], quote=True)
    display(HTML(f'<p><a href="{url}" target="_blank" rel="noopener">W&B Run を開く: {url}</a></p>'))
else:
    print(f"WANDB_MODE={run_info['mode']}: オンライン送信されていないため URL はありません。")
    print('オンライン記録する場合は WANDB_MODE=online に設定してから学習を実行してください。')

## 4. simulation video を確認する

Notebook で動画を確認した後、出力された W&B Run URL を開きます。Charts、Artifacts、Media が同じ Run にあることを確認してください。

In [ ]:
from IPython.display import Video, display
videos = sorted(isaaclab.glob('logs/rsl_rl/wandb-course-cartpole/*/videos/train/*.mp4'))
print(videos[-1] if videos else 'video not found')
display(Video(str(videos[-1]), embed=True)) if videos else None